# 🧠 LoRA-Seg : Fine-tuning Qwen2.5-1.5B pour la détection de frontières

## Objectif
Fine-tuner **Qwen2.5-1.5B-Instruct** avec LoRA pour classifier si un message
constitue une **frontière d'épisode** dans une conversation WhatsApp.

## Pipeline
1. Charger les données gold (tune/test) depuis Google Drive
2. Construire les prompts (fenêtre de 15 messages)
3. Fine-tuner avec LoRA (rank=16, 3 époques)
4. Évaluer sur le test set
5. Merger LoRA → modèle complet
6. Convertir en GGUF Q4_K_M pour CPU
7. Exporter le GGUF sur Google Drive

## Pré-requis
- Runtime **GPU T4** (gratuit sur Colab)
- Fichiers gold sur Drive : `My Drive/memory_ai_data/group_gold_tune.json` et `group_gold_test.json`

## Résultat attendu
- `qwen2.5-boundary-q4_k_m.gguf` (~1.1 GB) sur Google Drive
- Utilisable localement avec `llama-cpp-python` via `src/boundary_detector_llm.py`

In [ ]:
# ── CELLULE 1 : Installation des dépendances ─────────────────────────────
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    bitsandbytes>=0.43.0 accelerate>=0.30.0 datasets>=2.19.0 \
    sentencepiece protobuf scipy

# llama.cpp pour la conversion GGUF
!pip install -q llama-cpp-python

# Vérifier le GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu} ({vram:.1f} GB VRAM)')
else:
    raise RuntimeError('❌ Pas de GPU ! Va dans Runtime > Change runtime type > T4 GPU')

print('✓ Dépendances installées')

In [ ]:
# ── CELLULE 2 : Monter Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
assert os.path.isdir(DRIVE_DIR), f'❌ Dossier introuvable: {DRIVE_DIR}'

TUNE_PATH = os.path.join(DRIVE_DIR, 'group_gold_tune.json')
TEST_PATH = os.path.join(DRIVE_DIR, 'group_gold_test.json')

assert os.path.isfile(TUNE_PATH), f'❌ Fichier introuvable: {TUNE_PATH}'
assert os.path.isfile(TEST_PATH), f'❌ Fichier introuvable: {TEST_PATH}'

print(f'✓ Drive monté')
print(f'  tune: {TUNE_PATH}')
print(f'  test: {TEST_PATH}')

In [ ]:
# ── CELLULE 3 : Charger et parser les données gold ──────────────────────
import json
import re
from dataclasses import dataclass
from typing import List, Set

@dataclass
class Message:
    idx: int
    timestamp: str
    author: str
    text: str

def parse_whatsapp_line(line: str) -> dict | None:
    """Parse une ligne WhatsApp format: DD/MM/YYYY HH:MM - Author: Text"""
    m = re.match(r'(\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2})\s*-\s*([^:]+):\s*(.*)', line)
    if not m:
        return None
    return {'timestamp': m.group(1), 'author': m.group(2).strip(), 'text': m.group(3).strip()}

def load_gold(path: str) -> tuple[list[Message], set[int]]:
    """Charge un fichier gold et retourne (messages, boundary_indices)."""
    with open(path) as f:
        data = json.load(f)

    messages = []
    for i, art in enumerate(data['artifacts']):
        parsed = parse_whatsapp_line(art['content'])
        if parsed:
            messages.append(Message(idx=i, **parsed))
        else:
            # Messages système ou multilignes
            messages.append(Message(idx=i, timestamp='', author='système', text=art['content'][:200]))

    boundaries = set(data['boundaries'])
    return messages, boundaries

tune_msgs, tune_bounds = load_gold(TUNE_PATH)
test_msgs, test_bounds = load_gold(TEST_PATH)

print(f'✓ Tune: {len(tune_msgs)} messages, {len(tune_bounds)} frontières')
print(f'  Ratio positif: {len(tune_bounds)/len(tune_msgs)*100:.1f}%')
print(f'✓ Test: {len(test_msgs)} messages, {len(test_bounds)} frontières')
print(f'  Ratio positif: {len(test_bounds)/len(test_msgs)*100:.1f}%')

In [ ]:
# ── CELLULE 4 : Construire le dataset de prompts ────────────────────────
import random

WINDOW = 15  # messages de contexte avant le message cible
NEG_RATIO = 3  # sous-échantillonnage négatifs : 1 positif pour 3 négatifs

SYSTEM_PROMPT = """Tu es un expert en analyse conversationnelle. Ta tâche est de déterminer si le dernier message d'une conversation WhatsApp marque le DÉBUT d'un nouvel épisode thématique.

Un nouvel épisode commence quand:
- Le sujet de conversation change significativement
- Une nouvelle activité ou planification démarre
- Après une longue pause temporelle (plusieurs heures)
- Un nouveau participant relance sur un autre thème

Réponds UNIQUEMENT par un JSON: {"boundary": true} ou {"boundary": false}"""

def format_context(messages: list[Message], target_idx: int, window: int = WINDOW) -> str:
    """Formate la fenêtre de contexte autour du message cible."""
    start = max(0, target_idx - window)
    lines = []
    for msg in messages[start:target_idx]:
        lines.append(f'[{msg.timestamp}] {msg.author}: {msg.text}')

    # Message cible avec marqueur
    target = messages[target_idx]
    lines.append(f'\n>>> MESSAGE À ANALYSER >>>\n[{target.timestamp}] {target.author}: {target.text}')
    return '\n'.join(lines)

def build_dataset(messages: list[Message], boundaries: set[int], 
                  neg_ratio: int = NEG_RATIO, seed: int = 42) -> list[dict]:
    """Construit le dataset d'entraînement avec sous-échantillonnage."""
    random.seed(seed)

    positives = []
    negatives = []

    for i in range(1, len(messages)):  # skip index 0 (toujours début)
        context = format_context(messages, i)
        label = i in boundaries

        sample = {
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': context},
                {'role': 'assistant', 'content': json.dumps({'boundary': label})}
            ]
        }

        if label:
            positives.append(sample)
        else:
            negatives.append(sample)

    # Sous-échantillonner les négatifs
    n_neg = min(len(negatives), len(positives) * neg_ratio)
    negatives = random.sample(negatives, n_neg)

    dataset = positives + negatives
    random.shuffle(dataset)

    print(f'  Positifs: {len(positives)}, Négatifs: {n_neg}, Total: {len(dataset)}')
    print(f'  Ratio effectif: 1:{n_neg/len(positives):.1f}')
    return dataset

print('Construction du dataset tune...')
tune_dataset = build_dataset(tune_msgs, tune_bounds)

print('\nConstruction du dataset test...')
test_dataset = build_dataset(test_msgs, test_bounds, neg_ratio=100)  # garder tous les négatifs pour eval

In [ ]:
# ── CELLULE 5 : Charger Qwen2.5-1.5B + LoRA ─────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

# Quantisation 4-bit pour réduire la VRAM (T4 = 16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Charger le modèle
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

# Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'\n✓ Modèle chargé : {MODEL_ID}')
print(f'  Total params    : {total / 1e6:.1f}M')
print(f'  Trainable (LoRA): {trainable / 1e6:.1f}M ({100*trainable/total:.2f}%)')

In [ ]:
# ── CELLULE 6 : Préparer le dataset HuggingFace ─────────────────────────
from datasets import Dataset

def format_chat(sample):
    """Applique le chat template Qwen au sample."""
    text = tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

# Convertir en HF Dataset
train_ds = Dataset.from_list(tune_dataset).map(format_chat)
eval_ds = Dataset.from_list(test_dataset).map(format_chat)

# Stats sur les longueurs de tokens
lengths = [len(tokenizer.encode(t)) for t in train_ds['text'][:100]]
import statistics
print(f'✓ Train: {len(train_ds)} samples')
print(f'  Eval : {len(eval_ds)} samples')
print(f'  Token lengths (100 samples): median={statistics.median(lengths):.0f}, max={max(lengths)}, p95={sorted(lengths)[94]}')

In [ ]:
# ── CELLULE 7 : Fine-tuning avec SFTTrainer ─────────────────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/content/lora-boundary'

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    max_seq_length=1024,
    bf16=True,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_text_field='text',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    args=sft_config,
)

print(f'✓ Trainer configuré')
print(f'  Epochs: {sft_config.num_train_epochs}')
print(f'  Batch effectif: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}')
print(f'  LR: {sft_config.learning_rate}')
print(f'\n🚀 Lancement du fine-tuning...')

trainer.train()

# Sauvegarder les adaptateurs LoRA
trainer.save_model(os.path.join(OUTPUT_DIR, 'final'))
print(f'\n✓ Fine-tuning terminé ! Adaptateurs sauvés dans {OUTPUT_DIR}/final')

In [ ]:
# ── CELLULE 8 : Évaluation rapide sur le test set ───────────────────────
import re as re_module

def eval_boundary_detection(model, tokenizer, test_messages, test_boundaries, n_samples=200):
    """Évalue le modèle fine-tuné sur un échantillon du test set."""
    model.eval()
    indices = list(range(1, len(test_messages)))
    random.seed(42)
    sample_indices = random.sample(indices, min(n_samples, len(indices)))

    tp, fp, fn, tn = 0, 0, 0, 0

    for idx in sample_indices:
        context = format_context(test_messages, idx)
        messages_chat = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': context},
        ]

        input_text = tokenizer.apply_chat_template(
            messages_chat, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                temperature=0.1,
                do_sample=False,
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Parser la réponse
        predicted = False
        try:
            parsed = json.loads(response.strip())
            predicted = parsed.get('boundary', False)
        except json.JSONDecodeError:
            if 'true' in response.lower():
                predicted = True

        actual = idx in test_boundaries

        if predicted and actual:
            tp += 1
        elif predicted and not actual:
            fp += 1
        elif not predicted and actual:
            fn += 1
        else:
            tn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + fp + fn + tn)

    print(f'\n📊 Résultats sur {len(sample_indices)} samples:')
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1:        {f1:.4f}')
    print(f'  TP={tp} FP={fp} FN={fn} TN={tn}')

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

results = eval_boundary_detection(model, tokenizer, test_msgs, test_bounds)

In [ ]:
# ── CELLULE 9 : Merger LoRA dans le modèle de base ──────────────────────
from peft import AutoPeftModelForCausalLM

MERGED_DIR = '/content/qwen-boundary-merged'

print('Chargement du modèle avec adaptateurs LoRA...')
model_peft = AutoPeftModelForCausalLM.from_pretrained(
    os.path.join(OUTPUT_DIR, 'final'),
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print('Merge LoRA → modèle de base...')
merged_model = model_peft.merge_and_unload()

print(f'Sauvegarde du modèle mergé dans {MERGED_DIR}...')
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f'✓ Modèle mergé sauvé ({MERGED_DIR})')
!du -sh {MERGED_DIR}

In [ ]:
# ── CELLULE 10 : Conversion GGUF Q4_K_M ─────────────────────────────────
# Installer llama.cpp pour la conversion
!pip install -q huggingface_hub[hf_xet]

# Télécharger le script de conversion officiel
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama_cpp_repo 2>/dev/null || true
!pip install -q -r /content/llama_cpp_repo/requirements/requirements-convert_hf_to_gguf.txt 2>/dev/null || true

GGUF_F16 = '/content/qwen-boundary-f16.gguf'
GGUF_Q4 = '/content/qwen2.5-boundary-q4_k_m.gguf'

# Étape 1: HF → GGUF F16
print('\n📦 Conversion HF → GGUF F16...')
!python /content/llama_cpp_repo/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_F16} --outtype f16

# Étape 2: F16 → Q4_K_M
print('\n📦 Quantisation F16 → Q4_K_M...')
# Compiler llama-quantize si nécessaire
!cd /content/llama_cpp_repo && cmake -B build -DGGML_CUDA=OFF && cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -3
!/content/llama_cpp_repo/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M

import os
size_mb = os.path.getsize(GGUF_Q4) / 1e6
print(f'\n✓ GGUF créé: {GGUF_Q4} ({size_mb:.1f} MB)')

In [ ]:
# ── CELLULE 11 : Test rapide du GGUF ────────────────────────────────────
from llama_cpp import Llama

print('Chargement du GGUF pour test...')
llm = Llama(
    model_path=GGUF_Q4,
    n_ctx=1024,
    n_threads=2,
    verbose=False,
)

# Test avec un exemple du test set
test_context = format_context(test_msgs, 50)
test_messages_chat = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': test_context},
]

response = llm.create_chat_completion(
    messages=test_messages_chat,
    max_tokens=20,
    temperature=0.1,
)

answer = response['choices'][0]['message']['content']
actual = 50 in test_bounds

print(f'\n🧪 Test GGUF:')
print(f'  Réponse: {answer}')
print(f'  Attendu: boundary={actual}')
print(f'\n✓ Le GGUF fonctionne !')

del llm  # libérer la mémoire

In [ ]:
# ── CELLULE 12 : Exporter le GGUF sur Google Drive ──────────────────────
import shutil

DRIVE_OUTPUT = os.path.join(DRIVE_DIR, 'qwen2.5-boundary-q4_k_m.gguf')

print(f'Copie vers Google Drive...')
shutil.copy2(GGUF_Q4, DRIVE_OUTPUT)

size_mb = os.path.getsize(DRIVE_OUTPUT) / 1e6
print(f'\n✅ GGUF exporté : {DRIVE_OUTPUT} ({size_mb:.1f} MB)')
print(f'\n📋 Prochaines étapes :')
print(f'  1. Télécharger le GGUF depuis Google Drive')
print(f'  2. Le placer dans memory_ai_lab/models/')
print(f'  3. Utiliser avec boundary_detector_llm.py :')
print(f'     detector = LLMBoundaryDetector("models/qwen2.5-boundary-q4_k_m.gguf")')